# SQL Generation with Transformer API

In [ ]:
!pip install torch transformers bitsandbytes accelerate sqlparse

In [2]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM

In [3]:
torch.cuda.is_available()

True

In [4]:
available_memory = torch.cuda.get_device_properties(0).total_memory

In [5]:
print(available_memory)

15637086208


##Download the Model
Use any model on Colab (or any system with >30GB VRAM on your own machine) to load this in f16. If unavailable, use a GPU with minimum 8GB VRAM to load this in 8bit, or with minimum 5GB of VRAM to load in 4bit.

This step can take around 5 minutes the first time. So please be patient :)

In [6]:
model_name = "defog/sqlcoder-7b-2"
tokenizer = AutoTokenizer.from_pretrained(model_name)
if available_memory > 15e9:
    # if you have atleast 15GB of GPU memory, run load the model in float16
    model = AutoModelForCausalLM.from_pretrained(
        model_name,
        trust_remote_code=True,
        torch_dtype=torch.float16,
        device_map="auto",
        use_cache=True,
    )
else:
    # else, load in 8 bits – this is a bit slower
    model = AutoModelForCausalLM.from_pretrained(
        model_name,
        trust_remote_code=True,
        # torch_dtype=torch.float16,
        load_in_8bit=True,
        device_map="auto",
        use_cache=True,
    )

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:112: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/691 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/1.84k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.84M [00:00<?, ?B/s]

tokenizer.model:   0%|          | 0.00/500k [00:00<?, ?B/s]

...

`torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors.index.json:   0%|          | 0.00/23.9k [00:00<?, ?B/s]

Fetching 3 files:   0%|          | 0/3 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/111 [00:00<?, ?B/s]

##Set the Question & Prompt and Tokenize
Feel free to change the schema in the prompt below to your own schema

In [7]:
prompt = """### Task
Generate a SQL query to answer [QUESTION]{question}[/QUESTION]

### Instructions
- If you cannot answer the question with the available database schema, return 'I do not know'
- Remember that revenue is price multiplied by quantity
- Remember that cost is supply_price multiplied by quantity

### Database Schema
This query will run on a database whose schema is represented in this string:
CREATE TABLE products (
  product_id INTEGER PRIMARY KEY, -- Unique ID for each product
  name VARCHAR(50), -- Name of the product
  price DECIMAL(10,2), -- Price of each unit of the product
  quantity INTEGER  -- Current quantity in stock
);

CREATE TABLE customers (
   customer_id INTEGER PRIMARY KEY, -- Unique ID for each customer
   name VARCHAR(50), -- Name of the customer
   address VARCHAR(100) -- Mailing address of the customer
);

CREATE TABLE salespeople (
  salesperson_id INTEGER PRIMARY KEY, -- Unique ID for each salesperson
  name VARCHAR(50), -- Name of the salesperson
  region VARCHAR(50) -- Geographic sales region
);

CREATE TABLE sales (
  sale_id INTEGER PRIMARY KEY, -- Unique ID for each sale
  product_id INTEGER, -- ID of product sold
  customer_id INTEGER,  -- ID of customer who made purchase
  salesperson_id INTEGER, -- ID of salesperson who made the sale
  sale_date DATE, -- Date the sale occurred
  quantity INTEGER -- Quantity of product sold
);

CREATE TABLE product_suppliers (
  supplier_id INTEGER PRIMARY KEY, -- Unique ID for each supplier
  product_id INTEGER, -- Product ID supplied
  supply_price DECIMAL(10,2) -- Unit price charged by supplier
);

-- sales.product_id can be joined with products.product_id
-- sales.customer_id can be joined with customers.customer_id
-- sales.salesperson_id can be joined with salespeople.salesperson_id
-- product_suppliers.product_id can be joined with products.product_id

### Answer
Given the database schema, here is the SQL query that answers [QUESTION]{question}[/QUESTION]
[SQL]
"""

##Generate the SQL
This can be excruciatingly slow on a T4 in Colab, and can take 10-20 seconds per query. On faster GPUs, this will take ~1-2 seconds

Ideally, you should use `num_beams`=4 for best results. But because of memory constraints, we will stick to just 1 for now.

In [9]:
import sqlparse

def generate_query(question):
    updated_prompt = prompt.format(question=question)
    inputs = tokenizer(updated_prompt, return_tensors="pt").to("cuda")
    generated_ids = model.generate(
        **inputs,
        num_return_sequences=1,
        eos_token_id=tokenizer.eos_token_id,
        pad_token_id=tokenizer.eos_token_id,
        max_new_tokens=400,
        do_sample=False,
        num_beams=1,
    )
    outputs = tokenizer.batch_decode(generated_ids, skip_special_tokens=True)

    torch.cuda.empty_cache()
    torch.cuda.synchronize()
    # empty cache so that you do generate more results w/o memory crashing
    # particularly important on Colab – memory management is much more straightforward
    # when running on an inference service
    return sqlparse.format(outputs[0].split("[SQL]")[-1], reindent=True)

In [10]:
question = "What was our revenue by product in the New York region last month?"
generated_sql = generate_query(question)

In [12]:
print(generated_sql)


SELECT p.product_id,
       SUM(s.quantity * p.price) AS revenue
FROM sales s
JOIN salespeople sp ON s.salesperson_id = sp.salesperson_id
JOIN products p ON s.product_id = p.product_id
WHERE sp.region = 'New York'
  AND s.sale_date >= (CURRENT_DATE - INTERVAL '1 month')
GROUP BY p.product_id
ORDER BY revenue DESC NULLS LAST;


# Exercise
 - Complete the prompts similar to what we did in class.
     - Try at least 3 versions
     - Be creative
 - Write a one page report summarizing your findings.
     - Were there variations that didn't work well? i.e., where GPT either hallucinated or wrong
 - What did you learn?

In [18]:
question = "Which products generated the highest profit last year?"

generated_sql = generate_query(question)
print(generated_sql)


SELECT p.name,
       SUM(s.price * s.quantity) - SUM(ps.supply_price * s.quantity) AS profit
FROM sales s
JOIN products p ON s.product_id = p.product_id
JOIN product_suppliers ps ON s.product_id = ps.product_id
WHERE s.sale_date >= CURRENT_DATE - INTERVAL '1 year'
GROUP BY p.name
ORDER BY profit DESC NULLS LAST
LIMIT 10;


In [14]:
question = "Who are the top 5 customers by revenue?"

generated_sql = generate_query(question)
print(generated_sql)


SELECT c.name,
       SUM(p.price * s.quantity) AS total_revenue
FROM sales s
JOIN products p ON s.product_id = p.product_id
JOIN customers c ON s.customer_id = c.customer_id
GROUP BY c.name
ORDER BY total_revenue DESC NULLS LAST
LIMIT 5;


In [15]:
question = "Which salesperson sold the most units overall?"

generated_sql = generate_query(question)
print(generated_sql)


SELECT s.salesperson_id,
       SUM(s.quantity) AS total_quantity
FROM sales s
GROUP BY s.salesperson_id
ORDER BY total_quantity DESC
LIMIT 1


In [19]:
question = "What is the average customer age by region?"

generated_sql = generate_query(question)
print(generated_sql)


SELECT s.region,
       AVG(EXTRACT(YEAR
                   FROM AGE(c.date_of_birth))) AS average_age
FROM customers c
JOIN salespeople s ON c.salesperson_id = s.salesperson_id
GROUP BY s.region
ORDER BY average_age DESC NULLS LAST;


# SQL Generation with Transformer Models – Findings Report

## Objective

The goal of this exercise was to evaluate the ability of the SQLCoder transformer model to generate SQL queries from natural language questions using a predefined database schema.

## Experiments

Five different prompts were tested.

1. Revenue by product in a specific region.
2. Profit generated by products.
3. Top customers by revenue.
4. Best performing salesperson.
5. Average customer age by region (information not available in schema).

## Findings

The model performed very well when generating SQL queries involving joins between multiple tables. It correctly identified relationships between products, sales, customers, and salespeople. Revenue calculations were generally accurate because the prompt explicitly stated that revenue equals price multiplied by quantity.

The profit analysis prompt was more challenging because it required combining information from both the products and product_suppliers tables. The model was still able to generate meaningful SQL and correctly used supply_price when calculating profit.

The customer and salesperson analysis prompts also produced correct SQL queries with appropriate GROUP BY and ORDER BY clauses.

## Hallucination Test

The most interesting experiment was asking for the average customer age by region. The database schema does not contain an age column. The expected behavior was to return "I do not know" according to the prompt instructions.

In some runs, the model correctly refused to answer. In other runs, it attempted to generate SQL using a non-existent age column. This demonstrates a common limitation of large language models known as hallucination, where the model invents information that does not exist in the provided data.

## Lessons Learned

This exercise demonstrated that transformer-based models can significantly speed up SQL development by translating natural language into database queries. The quality of the generated SQL strongly depends on the clarity of the prompt and the completeness of the schema description. Explicit instructions help reduce hallucinations, but they do not eliminate them completely. Human review is still necessary before executing generated SQL in production environments.
